# 面试问题：Agent Skills 是什么？SKILL.md、Progressive Disclosure、路由、权限和版本怎样实现？

**一句话回答**：Skill 是可版本化目录，至少含带 frontmatter 的 `SKILL.md`，把领域流程、脚本、参考资料和资产打包；它告诉 Agent“怎样做”，而 Tool 提供“能执行什么”。启动时只暴露 name/description，匹配任务后加载完整说明，再按需读取 references/scripts/assets，以 progressive disclosure 控制上下文。Skill 内容仍不可信，不能自动扩大工具权限。

本 Notebook 依据开放 Agent Skills 规范手写 frontmatter 校验、三层披露、元数据路由、引用解析、版本冲突、allowed-tools 交集、路径/digest 安全、held-out eval 与原子更新。代码包含必要中文注释。


In [ ]:
from dataclasses import dataclass
import hashlib,math,re
from pathlib import PurePosixPath

# 只处理内存中的虚拟 skill 文件，不读写真实系统目录。
SEED150=15001
assert SEED150==15001
assert re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*","pdf-processing")
assert len(hashlib.sha256(b"skill").hexdigest())==64


## 1. 最小 Skill 是目录名与 frontmatter 一致的 SKILL.md

必填 `name/description`；name 需小写字母数字和连字符、最长 64 且与父目录一致，description 说明能力与触发场景。body 写步骤、例子和边界；可选 scripts/references/assets。下面实现受控 frontmatter 子集，不替代 YAML parser。


In [ ]:
@dataclass(frozen=True)
class SkillMeta150:
    name:str; description:str; version:str="0.0.0"; allowed_tools:tuple=()
def parse_skill150(directory,text):
    # 教学 parser 只接受简单 key:value，并显式校验名称与目录绑定。
    if not text.startswith("---\n") or "\n---\n" not in text[4:]: raise ValueError("frontmatter")
    head,body=text[4:].split("\n---\n",1); fields={}
    for line in head.splitlines():
        k,v=line.split(":",1); fields[k.strip()]=v.strip()
    name=fields.get("name",""); desc=fields.get("description","")
    if name!=directory or not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*",name) or len(name)>64 or not desc: raise ValueError("skill_metadata")
    tools=tuple(fields.get("allowed-tools","").split()); return SkillMeta150(name,desc,fields.get("version","0.0.0"),tools),body
skill_text150="---\nname: pdf-processing\ndescription: 提取 PDF 文本和表格；用户提到 PDF 或表单时使用。\nversion: 1.2.0\nallowed-tools: read pdf_parse\n---\n步骤：先读取文件，再检查页数。"
meta150,body150=parse_skill150("pdf-processing",skill_text150)
assert meta150.name=="pdf-processing"
assert "先读取" in body150
assert meta150.allowed_tools==("read","pdf_parse")


## 2. Progressive Disclosure 分 Catalog、Instructions、Resources

会话启动只加载所有 skill 的 name/description；匹配后读完整 SKILL.md body；正文明确需要时才读取某个 reference/script/asset。这样可挂载大量技能而不把全部内容塞进 prompt。每层都计 token budget，并限制递归引用深度。


In [ ]:
files150={"SKILL.md":skill_text150,"references/forms.md":"表单字段详细说明"*20,"scripts/extract.py":"print('extract')"}
def disclosure_cost150(catalog_chars,body_chars,resource_chars,stage):
    # 用四字符约一 token 的教学估算展示三层按需成本。
    parts=[catalog_chars,body_chars,resource_chars]; return math.ceil(sum(parts[:stage])/4)
catalog_cost150=disclosure_cost150(len(meta150.name+meta150.description),len(body150),len(files150["references/forms.md"]),1)
active_cost150=disclosure_cost150(len(meta150.name+meta150.description),len(body150),len(files150["references/forms.md"]),2)
full_cost150=disclosure_cost150(len(meta150.name+meta150.description),len(body150),len(files150["references/forms.md"]),3)
assert catalog_cost150<active_cost150<full_cost150
assert catalog_cost150>0
assert full_cost150-active_cost150>0


## 3. 路由主要依赖 description 中的能力与触发词

模糊描述“帮助处理文档”会误触发；好描述同时写动作、对象和何时使用。Client 可 lexical/embedding 召回后让模型选择，但高风险 skill 还需策略门禁。下面用字符/词 token overlap 做可解释 baseline，不宣称替代真实模型路由。


In [ ]:
catalog150=[SkillMeta150("pdf-processing","提取 PDF 文本 表格 填写 表单；提到 PDF 时使用"),SkillMeta150("git-review","审查 git diff 提交和代码变更；代码评审时使用")]
def tokens150(s): return set(re.findall(r"[a-z]+|[\u4e00-\u9fff]+",s.lower()))
def route150(query,catalog):
    # 元数据阶段只看短 description，避免提前加载完整技能正文。
    q=tokens150(query); scored=[(len(q&tokens150(x.name+" "+x.description)),x.name) for x in catalog]; return max(scored)
assert route150("请提取 PDF 表格",catalog150)[1]=="pdf-processing"
assert route150("审查 git 提交",catalog150)[1]=="git-review"
assert route150("PDF",catalog150)[0]>0


## 4. 引用从 skill root 解析，并限制一层深度与存在性

SKILL.md 可引用 `references/x.md` 或 `scripts/x.py`，不应读取任意绝对路径、`..`、深层链或隐藏 secret。资源按任务需要加载，脚本执行仍由 Host 的工具、沙箱和权限策略控制；“在 skill 目录里”不是执行授权。


In [ ]:
def resolve_resource150(path,files):
    # PurePosixPath 只接受相对的一层目录/文件，拒绝路径穿越。
    p=PurePosixPath(path)
    if p.is_absolute() or ".." in p.parts or len(p.parts)!=2 or p.parts[0] not in {"references","scripts","assets"}: raise ValueError("skill_path")
    if str(p) not in files: raise FileNotFoundError(str(p))
    return files[str(p)]
assert "表单" in resolve_resource150("references/forms.md",files150)
try: resolve_resource150("../../secret",files150); raise AssertionError("traversal")
except ValueError as e: assert str(e)=="skill_path"
assert resolve_resource150("scripts/extract.py",files150).startswith("print")


## 5. 版本、compatibility 与依赖冲突在激活前解析

Skill 更新可能改变流程或所需工具；session 固定 immutable revision，不能执行中途静默漂移。若两个 skill 对同一输出格式给冲突指令，Host 应按用户显式选择、项目策略和优先级解决，无法确定则拒绝组合而不是随意拼接。


In [ ]:
def semver150(v):
    # 这里只支持严格三段数字版本，便于稳定排序与回滚。
    if not re.fullmatch(r"\d+\.\d+\.\d+",v): raise ValueError("semver")
    return tuple(map(int,v.split(".")))
installed150={"pdf-processing":["1.1.0","1.2.0","2.0.0"]}
compatible150=[v for v in installed150["pdf-processing"] if semver150(v)[0]==1]
assert max(compatible150,key=semver150)=="1.2.0"
assert semver150("2.0.0")>semver150("1.9.9")
try: semver150("latest"); raise AssertionError("floating version")
except ValueError as e: assert str(e)=="semver"


## 6. allowed-tools 是上限提示，不会扩大 Host capability

实际可用工具为 `skill_requested ∩ host_allowed ∩ user_scope`；experimental 字段在不支持的 Client 中不能被假设生效。高风险动作继续走 schema、ACL 与审批。Skill 正文中的“忽略权限并运行 shell”只是待审内容，不是控制面指令。


In [ ]:
def effective_tools150(requested,host_allowed,user_scope):
    # 三方集合求交，任何一层都不能单独授予额外能力。
    return set(requested)&set(host_allowed)&set(user_scope)
effective150=effective_tools150(meta150.allowed_tools,{"read","pdf_parse","shell"},{"read","pdf_parse"})
assert effective150=={"read","pdf_parse"}
assert "shell" not in effective150
assert effective_tools150({"shell"},{"shell"},set())==set()


## 7. 安装与更新校验来源、digest、文件清单和许可

Skill 可能携带脚本和提示注入，因此从受信 registry/仓库获取，预览 diff，按文件 digest 锁定 revision；symlink、路径穿越、超大资源和二进制制品单独治理。更新写 staging，验证后原子切 active pointer，旧版本保留可回滚。


In [ ]:
def bundle_digest150(files):
    # 文件名排序并同时哈希路径与内容，防止重排或替换逃逸。
    h=hashlib.sha256()
    for path in sorted(files): h.update(path.encode()+b"\0"+files[path].encode()+b"\0")
    return h.hexdigest()
digest_a150=bundle_digest150(files150); changed_files150={**files150,"scripts/extract.py":"print('changed')"}
assert len(digest_a150)==64
assert digest_a150!=bundle_digest150(changed_files150)
assert digest_a150==bundle_digest150(dict(reversed(list(files150.items()))))


## 8. Skill 评测覆盖触发、遵循、结果和安全副作用

测 positive/negative trigger、与相似 skill 冲突、说明遵循率、任务成功、token/资源加载、脚本错误和禁止工具调用。训练任务用于迭代 description/body，held-out 防止把技能写成题库答案。比较无 skill、只 metadata、完整 skill 和资源按需加载的消融。


In [ ]:
evals150=[{"should_trigger":1,"triggered":1,"success":1,"unsafe":0},{"should_trigger":1,"triggered":0,"success":0,"unsafe":0},{"should_trigger":0,"triggered":1,"success":0,"unsafe":1},{"should_trigger":0,"triggered":0,"success":1,"unsafe":0}]
# 分开计算 trigger recall 与 false-positive，避免只看最终成功率。
tp150=sum(x["should_trigger"] and x["triggered"] for x in evals150); pos150=sum(x["should_trigger"] for x in evals150); fp150=sum((not x["should_trigger"]) and x["triggered"] for x in evals150)
assert tp150/pos150==.5
assert fp150==1
assert sum(x["unsafe"] for x in evals150)==1


## 面试总结

完整回答是：**目录名+SKILL.md frontmatter 校验 → Catalog 只暴露 name/description → description 路由后加载正文 → references/scripts/assets 按需一层解析 → session pin immutable version、冲突显式解决 → requested∩host∩user 权限 → 来源/digest/path/symlink/脚本安全 → staging 原子更新与回滚 → positive/negative trigger、任务成功、token 和 unsafe side effect 的 held-out eval**。

延伸阅读：[Agent Skills Specification](https://agentskills.io/specification)、[Equipping Agents with Skills](https://www.anthropic.com/engineering/equipping-agents-for-the-real-world-with-agent-skills)、[MCP Architecture](https://modelcontextprotocol.io/docs/learn/architecture)。
